# Portfolio Summary Mart Validation

Objective: Validate that `mart.portfolio_summary` accurately aggregates the feature layer without loss, duplication, or distortion and produces meaningful portfolio-level business insights.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.width", None)

DB_PATH = (
    Path.cwd().parent.parent
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

print(DB_PATH)

conn = duckdb.connect(str(DB_PATH))

d:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


## Mart Row Count Validation

In [2]:
# =====================================================
# MART ROW COUNT
# =====================================================

conn.execute("""
select
    count(*) as mart_rows
from mart.portfolio_summary
""").fetchdf()

,mart_rows
0,160428


### Finding

The mart contains **160,428 aggregated rows**. This is a realistic reduction from 2.26 million loan-level records and reflects aggregation across month, grade, state, and purpose combinations.

**Decision:** Row-count validation passed.

## Mart Grain Validation

In [3]:
# =====================================================
# MART GRAIN VALIDATION
# =====================================================

conn.execute("""
select

    count(*) as rows,

    count(
        distinct concat(
            cast(issue_month as varchar),
            '|',
            grade,
            '|',
            addr_state,
            '|',
            purpose
        )
    ) as unique_rows

from mart.portfolio_summary
""").fetchdf()

,rows,unique_rows
0,160428,160428


### Finding

The mart contains **160,428 rows and 160,428 unique grain combinations**.

No duplicate aggregations exist.

**Implication:** The grain `(issue_month, grade, addr_state, purpose)` is uniquely enforced.

**Decision:** Grain validation passed.

## Feature-to-Mart Reconciliation

In [4]:
# =====================================================
# FEATURE TO MART RECONCILIATION
# =====================================================

conn.execute("""
select

    (
        select count(*)
        from feature.loan_features_v1
    ) as feature_rows,

    (
        select sum(loan_count)
        from mart.portfolio_summary
    ) as mart_loan_count
""").fetchdf()

,feature_rows,mart_loan_count
0,2260668,2260668.0


### Finding

The feature layer contains **2,260,668 loans** and the mart aggregates to the same total loan count.

**Implication:** No loans were lost during aggregation.

**Decision:** Reconciliation passed.

## Mart Structure Review

In [5]:
# =====================================================
# MART STRUCTURE
# =====================================================

schema = conn.execute("""
describe mart.portfolio_summary
""").fetchdf()

print(f"Columns: {len(schema)}")

schema

Columns: 17


,column_name,column_type,null,key,default,extra
0,issue_month,DATE,YES,None,None,None
1,grade,VARCHAR,YES,None,None,None
2,addr_state,VARCHAR,YES,None,None,None
3,purpose,VARCHAR,YES,None,None,None
4,loan_count,BIGINT,YES,None,None,None
5,total_loan_amount,DOUBLE,YES,None,None,None
6,avg_loan_amount,DOUBLE,YES,None,None,None
7,avg_interest_rate,DOUBLE,YES,None,None,None
8,avg_fico,DOUBLE,YES,None,None,None
9,avg_dti,DOUBLE,YES,None,None,None


### Finding

The mart contains **17 columns** covering dimensions, volume metrics, risk metrics, utilization metrics, and default metrics.

Notably, the mart now includes `defaulted_loan_amount`, enabling exposure-based risk reporting in addition to count-based reporting.

**Decision:** Schema is sufficient for executive portfolio reporting.

## Default Rate Reconciliation

In [6]:
# =====================================================
# DEFAULT RATE RECONCILIATION
# =====================================================

conn.execute("""
select

    (
        select round(
            100.0 * avg(default_flag),
            4
        )
        from feature.loan_features_v1
    ) as feature_default_rate,

    (
        select round(
            sum(default_count) * 100.0 /
            sum(loan_count),
            4
        )
        from mart.portfolio_summary
    ) as mart_default_rate
""").fetchdf()

,feature_default_rate,mart_default_rate
0,12.8646,12.8646


### Finding

Feature-layer default rate = **12.8646%**.

Mart default rate = **12.8646%**.

The values match exactly.

**Decision:** Default aggregation logic is correct.

## High Utilization Reconciliation

In [7]:
# =====================================================
# HIGH UTILIZATION RECONCILIATION
# =====================================================

conn.execute("""
select

    (
        select round(
            100.0 *
            avg(high_utilization_flag),
            4
        )
        from feature.loan_features_v1
    ) as feature_high_util_rate,

    (
        select round(
            sum(high_utilization_count)
            * 100.0 /
            sum(loan_count),
            4
        )
        from mart.portfolio_summary
    ) as mart_high_util_rate
""").fetchdf()

,feature_high_util_rate,mart_high_util_rate
0,13.7569,13.7569


### Finding

Feature-layer high-utilization rate = **13.7569%**.

Mart high-utilization rate = **13.7569%**.

The values match exactly.

**Decision:** Utilization aggregation logic is correct.

## Date Coverage Validation

In [8]:
# =====================================================
# DATE RANGE VALIDATION
# =====================================================

conn.execute("""
select

    min(issue_month) as min_issue_month,

    max(issue_month) as max_issue_month,

    count(
        distinct issue_month
    ) as unique_months

from mart.portfolio_summary
""").fetchdf()

,min_issue_month,max_issue_month,unique_months
0,2007-06-01,2018-12-01,139


### Finding

The mart spans **June 2007 through December 2018** and contains **139 unique months**.

**Implication:** The full LendingClub observation window is represented in the mart.

**Decision:** Date coverage validation passed.

## Geographic Concentration Analysis

In [9]:
# =====================================================
# TOP STATES
# =====================================================

conn.execute("""
select

    addr_state,

    sum(loan_count) as loans,

    round(
        100.0 *
        sum(loan_count) /
        (
            select sum(loan_count)
            from mart.portfolio_summary
        ),
        2
    ) as portfolio_share_pct

from mart.portfolio_summary

group by addr_state

order by loans desc

limit 15
""").fetchdf()

,addr_state,loans,portfolio_share_pct
0,CA,314533.0,13.91
1,NY,186389.0,8.24
2,TX,186335.0,8.24
3,FL,161991.0,7.17
4,IL,91173.0,4.03
5,NJ,83132.0,3.68
6,PA,76939.0,3.40
7,OH,75132.0,3.32
8,GA,74196.0,3.28
9,VA,62954.0,2.78


### Finding

California dominates the portfolio with **314,533 loans (13.91%)**.

New York and Texas each contribute approximately **8.24%** of all loans, followed by Florida at **7.17%**.

The top four states account for more than one-third of the entire portfolio.

**Implication:** Geographic concentration is material and should be monitored in downstream risk reporting.

## Purpose Distribution Analysis

In [10]:
# =====================================================
# TOP PURPOSES
# =====================================================

conn.execute("""
select

    purpose,

    sum(loan_count) as loans,

    round(
        100.0 *
        sum(loan_count) /
        (
            select sum(loan_count)
            from mart.portfolio_summary
        ),
        2
    ) as portfolio_share_pct

from mart.portfolio_summary

group by purpose

order by loans desc
""").fetchdf()

,purpose,loans,portfolio_share_pct
0,debt_consolidation,1277877.0,56.53
1,credit_card,516971.0,22.87
2,home_improvement,150457.0,6.66
3,other,139440.0,6.17
4,major_purchase,50445.0,2.23
5,medical,27488.0,1.22
6,small_business,24689.0,1.09
7,car,24013.0,1.06
8,vacation,15525.0,0.69
9,moving,15403.0,0.68


### Finding

**Debt Consolidation** is the primary use case, representing **56.53%** of all loans.

**Credit Card** refinancing contributes another **22.87%**.

Together, these two purposes account for nearly **80%** of LendingClub originations.

**Implication:** Portfolio performance will be heavily driven by consumer debt refinancing behaviour.

## Grade Distribution and Risk Analysis

In [11]:
# =====================================================
# GRADE DISTRIBUTION
# =====================================================

conn.execute("""
select

    grade,

    sum(loan_count) as loans,

    round(
        100.0 *
        sum(loan_count) /
        (
            select sum(loan_count)
            from mart.portfolio_summary
        ),
        2
    ) as portfolio_share_pct,

    round(
        sum(default_count)
        * 100.0 /
        sum(loan_count),
        2
    ) as default_rate

from mart.portfolio_summary

group by grade

order by grade
""").fetchdf()

,grade,loans,portfolio_share_pct,default_rate
0,A,433027.0,19.15,3.59
1,B,663557.0,29.35,8.66
2,C,650053.0,28.75,14.36
3,D,324424.0,14.35,20.35
4,E,135639.0,6.00,28.28
5,F,41800.0,1.85,36.42
6,G,12168.0,0.54,40.01


### Finding

Grades **B (29.35%)** and **C (28.75%)** represent the majority of the portfolio.

Default rates increase monotonically:

- A: 3.59%
- B: 8.66%
- C: 14.36%
- D: 20.35%
- E: 28.28%
- F: 36.42%
- G: 40.01%

**Implication:** Risk ordering behaves exactly as expected.

**Decision:** Grade-risk validation passed.

## Portfolio-Level Metrics

In [12]:
# =====================================================
# PORTFOLIO METRICS
# =====================================================

conn.execute("""
select

    sum(loan_count) as loans,

    round(
        sum(total_loan_amount),
        0
    ) as total_loan_amount,

    round(
        avg(avg_interest_rate),
        2
    ) as avg_interest_rate,

    round(
        avg(avg_fico),
        2
    ) as avg_fico,

    round(
        avg(avg_dti),
        2
    ) as avg_dti,

    round(
        avg(avg_revolving_utilization_ratio),
        2
    ) as avg_revolving_utilization

from mart.portfolio_summary
""").fetchdf()

,loans,total_loan_amount,avg_interest_rate,avg_fico,avg_dti,avg_revolving_utilization
0,2260668.0,3.401612e+10,15.21,702.51,17.65,48.88


### Finding

Portfolio summary statistics:

- Loans: **2.26 million**
- Total funded amount: **$34.0 billion**
- Average interest rate: **15.21%**
- Average FICO: **702.51**
- Average DTI: **17.65**
- Average revolving utilization: **48.88%**

These values are consistent with a large unsecured consumer lending portfolio.

## Sample Record Review

In [13]:
# =====================================================
# SAMPLE RECORDS
# =====================================================

conn.execute("""
select *
from mart.portfolio_summary
order by issue_month desc
limit 20
""").fetchdf()

,issue_month,grade,addr_state,purpose,loan_count,total_loan_amount,avg_loan_amount,avg_interest_rate,avg_fico,avg_dti,avg_loan_to_income_ratio,avg_revolving_utilization_ratio,default_count,defaulted_loan_amount,default_rate,high_utilization_count,high_utilization_rate
0,2018-12-01,E,IN,other,7,56150.0,8021.428571,24.785714,692.714286,23.592857,0.179019,52.928571,1.0,9100.0,14.29,1.0,14.29
1,2018-12-01,D,MD,debt_consolidation,66,1170000.0,17727.272727,19.149394,689.045455,20.106515,0.217597,57.845455,1.0,5000.0,1.52,14.0,21.21
2,2018-12-01,A,IL,credit_card,175,2840200.0,16229.714286,7.724229,712.914286,16.913543,0.196580,42.753143,0.0,0.0,0.00,18.0,10.29
3,2018-12-01,C,MA,credit_card,61,981675.0,16093.032787,15.014262,687.737705,18.061148,0.252602,53.980328,0.0,0.0,0.00,10.0,16.39
4,2018-12-01,A,MI,other,8,80400.0,10050.000000,7.836250,747.000000,21.073750,0.116178,22.612500,0.0,0.0,0.00,0.0,0.00
5,2018-12-01,D,MD,home_improvement,9,134375.0,14930.555556,19.267778,675.888889,19.354444,0.142434,59.188889,0.0,0.0,0.00,2.0,22.22
6,2018-12-01,A,PA,debt_consolidation,189,3214250.0,17006.613757,7.810053,732.793651,19.280635,0.222924,33.806878,1.0,20975.0,0.53,3.0,1.59
7,2018-12-01,D,CA,credit_card,130,2136100.0,16431.538462,19.519615,686.192308,22.402713,0.276215,57.116154,1.0,35000.0,0.77,27.0,20.77
8,2018-12-01,B,NE,credit_card,21,350500.0,16690.476190,11.263333,695.809524,19.864286,0.228198,53.614286,0.0,0.0,0.00,4.0,19.05
9,2018-12-01,A,MO,credit_card,55,895850.0,16288.181818,7.586364,713.000000,16.398364,0.223088,44.727273,1.0,3600.0,1.82,2.0,3.64


### Finding

Sample records confirm that the mart correctly stores aggregated observations at the target grain.

Metrics, rates, and dimensional attributes appear internally consistent.

**Decision:** Spot-check review passed.

# Validation Summary

## Validation Result

Status: PASSED

The Portfolio Summary Mart successfully passed all structural, reconciliation, and business validation checks.

## Technical Validation Results

| Validation Area | Result |
|----------|----------|
| Mart Creation | Passed |
| Row Count Validation | Passed |
| Grain Uniqueness Validation | Passed |
| Feature-to-Mart Reconciliation | Passed |
| Default Rate Reconciliation | Passed |
| High Utilization Reconciliation | Passed |
| Date Range Validation | Passed |
| Schema Validation | Passed |

## Mart Characteristics

| Metric | Value |
|----------|----------|
| Rows | 160,428 |
| Columns | 16 |
| Source Table | feature.loan_features_v1 |
| Mart Grain | issue_month × grade × addr_state × purpose |
| Observation Period | Jun-2007 to Dec-2018 |
| Portfolio Loans Represented | 2,260,668 |

## Key Portfolio Findings

### Portfolio Composition

- The mart aggregates the entire LendingClub portfolio without loss of records.
- Loan counts reconcile exactly to the feature layer population of 2.26 million loans.
- No duplicate mart records were identified.

### Geographic Concentration

- California is the largest state exposure in the portfolio.
- New York, Texas, and Florida are also significant contributors.
- Portfolio activity is concentrated in a relatively small number of large states.

### Purpose Concentration

- Debt Consolidation is the dominant lending purpose.
- Credit Card refinancing represents the second-largest use case.
- The majority of LendingClub activity is concentrated in debt restructuring rather than discretionary borrowing.

### Credit Quality Distribution

- Grades B and C represent the largest share of the portfolio.
- Grade A borrowers exhibit the lowest observed default rates.
- Default rates increase consistently through Grade G.

### Risk Behaviour

- Default-rate ordering follows the expected LendingClub credit hierarchy.
- High-risk grades exhibit materially higher default frequencies than prime grades.
- Risk segmentation remains preserved after aggregation into the mart layer.

## Conclusion

`mart.portfolio_summary` is a validated analytical mart built on top of the LendingClub feature layer.

The mart preserves full portfolio coverage, maintains the intended grain, reconciles exactly to the source feature store, and produces economically consistent risk metrics.

The table is suitable for executive reporting, portfolio monitoring, dashboard development, and downstream analytical marts.

## Next Step

Proceed to:

**M002 – Grade Performance Mart**

The Grade Performance Mart will provide a dedicated view of portfolio composition, risk, pricing, utilization, and default behaviour across LendingClub credit grades.